# IBPL video prototype: McByte++ first clip

This notebook tracks the selected YouTube interval **154-174 seconds** from Hapoel Eilat vs Bnei Herzliya. It uses McByte++ with its supplied SportsMOT YOLOX detector and online re-identification.

Before running: choose **Runtime -> Change runtime type -> T4 GPU** (or another GPU), then run cells in order. The notebook does not modify the production app or database. A free Colab GPU is enough to attempt this 20-second feasibility test, subject to availability.


In [ ]:
# Configuration
from pathlib import Path
VIDEO_URL = 'https://www.youtube.com/watch?v=urrtcQUzwmo'
START_SECONDS, END_SECONDS = 154, 174
LIVE_END_IN_CLIP = 19.436541  # detected close-up cut; inspect before changing
PROCESS_FPS = 25
WORK = Path('/content/ibpl_mcbyte')
REPO = WORK / 'McBytePlusPlus'
VENV = WORK / 'venv'
INPUT = WORK / 'input'
RESULTS = WORK / 'results'
for p in (WORK, INPUT, RESULTS): p.mkdir(parents=True, exist_ok=True)
print({'url': VIDEO_URL, 'range': [START_SECONDS, END_SECONDS], 'process_fps': PROCESS_FPS})


In [ ]:
# Hardware preflight: stop here if NVIDIA GPU is not visible.
import shutil, subprocess
assert shutil.which('nvidia-smi'), 'No GPU runtime. Select a GPU in Runtime settings and reconnect.'
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['gcc', '--version'], check=True)
subprocess.run(['ffmpeg', '-version'], check=True, stdout=subprocess.PIPE)


## Install isolated Python 3.10 environment
McByte++ documents Python 3.10, PyTorch 2.9.1/CUDA 12.8 and NumPy 1.24.4. This may take several minutes. Rerunning completed install cells is normally harmless.


In [ ]:
# Install uv and create the documented Python version without altering Colab's kernel.
subprocess.run(['pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.10'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.10', str(VENV)], check=True)
PY = str(VENV / 'bin' / 'python')
print(subprocess.check_output([PY, '--version'], text=True).strip())


In [ ]:
# Clone once and record the exact upstream commit used.
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/tstanczyk95/McBytePlusPlus.git', str(REPO)], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('McByte++ commit:', commit)


In [ ]:
# Install the tracker following upstream's documented order.
def uv_install(*args):
    subprocess.run(['uv', 'pip', 'install', '--python', PY, *args], check=True)
uv_install('torch==2.9.1', 'torchvision==0.24.1', 'torchaudio==2.9.1', '--index-url', 'https://download.pytorch.org/whl/cu128')
uv_install('-r', str(REPO / 'requirements.txt'))
uv_install('-e', str(REPO), '--no-build-isolation')
uv_install('cython', 'cython_bbox', 'pycocotools', 'gdown', 'yt-dlp', 'numpy==1.24.4')
subprocess.run(['uv', 'pip', 'install', '--python', PY, '-e', str(REPO / 'mask_propagation' / 'EdgeTAM')], check=True)
uv_install('-r', str(REPO / 'deep-person-reid' / 'requirements.txt'))
uv_install('setuptools==59.8.0', 'numpy==1.24.4')
subprocess.run([PY, 'setup.py', 'develop'], cwd=REPO / 'deep-person-reid', check=True)


In [ ]:
# Download upstream's checkpoint backup folder and place required files.
import os
MODEL_DOWNLOAD = WORK / 'model_download'
MODEL_DOWNLOAD.mkdir(exist_ok=True)
required = {
    'yolox_x_sports_mix.pth.tar': REPO / 'pretrained' / 'yolox_x_sports_mix.pth.tar',
    'edgetam.pt': REPO / 'mask_propagation' / 'EdgeTAM' / 'checkpoints' / 'edgetam.pt',
    'sports_model.pth.tar-60': REPO / 'deep-person-reid' / 'pretrained' / 'sports_model.pth.tar-60',
}
if not all(p.exists() for p in required.values()):
    subprocess.run([PY, '-m', 'gdown', '--folder', 'https://drive.google.com/drive/folders/1yzzJk9dpJUY3lIHdkkFyGtKL2F-FenN6?usp=sharing', '-O', str(MODEL_DOWNLOAD)], check=True)
    found = {p.name: p for p in MODEL_DOWNLOAD.rglob('*') if p.is_file()}
    for name, target in required.items():
        if not target.exists():
            assert name in found, f'Missing checkpoint {name}; inspect the upstream model folder.'
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(found[name], target)
for name, path in required.items(): print(name, path.stat().st_size, 'bytes')


## Retrieve and prepare the selected clip
The source clip remains 154-174 seconds. Tracking uses the live camera shot before the detected close-up and resamples it to 25 FPS. This makes the first run faster and gives the tracker its actual processed frame rate.


In [ ]:
SOURCE = INPUT / 'source.mp4'
FRAMES = INPUT / 'shot_001_frames_25fps'
if not SOURCE.exists():
    template = str(INPUT / 'download.%(ext)s')
    subprocess.run([PY, '-m', 'yt_dlp', '--no-playlist', '--download-sections', f'*{START_SECONDS}-{END_SECONDS}', '--force-keyframes-at-cuts', '-f', 'bv*[height<=1080]+ba/b[height<=1080]', '--merge-output-format', 'mp4', '-o', template, VIDEO_URL], check=True)
    downloaded = sorted(INPUT.glob('download.*'))
    assert downloaded, 'Clip download did not create a media file.'
    downloaded[0].replace(SOURCE)
FRAMES.mkdir(exist_ok=True)
if not any(FRAMES.glob('*.jpg')):
    subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-nostdin', '-i', str(SOURCE), '-t', str(LIVE_END_IN_CLIP), '-vf', f'fps={PROCESS_FPS}', '-q:v', '2', str(FRAMES / '%06d.jpg')], check=True)
frame_count = len(list(FRAMES.glob('*.jpg')))
assert 480 <= frame_count <= 490, f'Unexpected frame count: {frame_count}'
print('Prepared frames:', frame_count)


In [ ]:
# Validate CUDA and imports before the timed run. Print upstream errors in full.
import os
MCBYTE_ENV = os.environ.copy()
MCBYTE_ENV['MPLBACKEND'] = 'Agg'  # Colab's inline backend is unavailable in the isolated venv
check = "import torch,numpy,cv2; print('torch',torch.__version__,'cuda',torch.version.cuda,'gpu',torch.cuda.get_device_name(0),'vram_GB',round(torch.cuda.get_device_properties(0).total_memory/2**30,2),'numpy',numpy.__version__)"
subprocess.run([PY, '-c', check], cwd=REPO, check=True, env=MCBYTE_ENV)
probe = subprocess.run([PY, 'tools/demo_track__with_reid.py', '--help'], cwd=REPO, text=True, capture_output=True, env=MCBYTE_ENV)
print(probe.stdout)
if probe.returncode:
    print('--- McByte++ import error (stderr) ---')
    print(probe.stderr)
    raise RuntimeError(f'McByte++ preflight failed with exit code {probe.returncode}; use the final traceback above for diagnosis')
print('McByte++ preflight passed.')


## Run tracking
This is the long cell. Runtime depends on the allocated GPU. If it fails with GPU out-of-memory, capture the error; do not repeatedly lower quality without recording the failed configuration.


In [ ]:
import json, threading, time
memory_samples = []
stop_poll = False
def poll_gpu():
    while not stop_poll:
        try:
            used = int(subprocess.check_output(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'], text=True).splitlines()[0])
            memory_samples.append(used)
        except Exception:
            pass
        time.sleep(1)
thread = threading.Thread(target=poll_gpu, daemon=True); thread.start()
cmd = [PY, 'tools/demo_track__with_reid.py', '--path', str(FRAMES), '--fps', str(PROCESS_FPS), '--vis_type', 'basic']
started = time.perf_counter(); error = None
try:
    subprocess.run(cmd, cwd=REPO, check=True, env=MCBYTE_ENV)
except subprocess.CalledProcessError as exc:
    error = f'process_exit_{exc.returncode}'
finally:
    elapsed = time.perf_counter() - started; stop_poll = True; thread.join(timeout=2)
benchmark = {'mcbyte_commit': commit, 'source_range_seconds': [START_SECONDS, END_SECONDS], 'tracked_clip_seconds': LIVE_END_IN_CLIP, 'input_frames': frame_count, 'process_fps': PROCESS_FPS, 'wall_seconds': elapsed, 'effective_input_fps': frame_count / elapsed, 'peak_sampled_gpu_memory_mib': max(memory_samples) if memory_samples else None, 'status': 'failed' if error else 'completed', 'error': error}
(RESULTS / 'benchmark.json').write_text(json.dumps(benchmark, indent=2) + '\n')
print(json.dumps(benchmark, indent=2))
assert error is None, 'Tracking failed; benchmark.json records the attempt.'


In [ ]:
# Locate the new output, create an annotated MP4 and summarize tracks.
import csv
track_root = REPO / 'YOLOX_outputs' / 'yolox_x_mix_det' / 'track_vis'
frame_dirs = [p for p in track_root.iterdir() if p.is_dir() and any(p.glob('*.jpg'))]
assert frame_dirs, f'No annotated frame directory found under {track_root}'
annotated_frames = max(frame_dirs, key=lambda p: p.stat().st_mtime)
mot_candidates = sorted(track_root.glob('*.txt'), key=lambda p: p.stat().st_mtime)
assert mot_candidates, 'No MOT text output found.'
mot_file = mot_candidates[-1]
annotated_mp4 = RESULTS / 'tracked_shot_001.mp4'
subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-nostdin', '-y', '-framerate', str(PROCESS_FPS), '-i', str(annotated_frames / '%06d.jpg'), '-c:v', 'libx264', '-pix_fmt', 'yuv420p', str(annotated_mp4)], check=True)
shutil.copy2(mot_file, RESULTS / 'tracks_mot.txt')
rows = list(csv.reader((RESULTS / 'tracks_mot.txt').open()))
summary = {'observation_rows': len(rows), 'track_ids': sorted({int(r[1]) for r in rows}), 'frames_with_tracks': len({int(r[0]) for r in rows}), 'annotated_video': str(annotated_mp4)}
(RESULTS / 'tracking_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))


In [ ]:
# Preview the result inside Colab.
from IPython.display import Video, display
display(Video(str(RESULTS / 'tracked_shot_001.mp4'), embed=True, width=960))


In [ ]:
# Save a compact result bundle to Google Drive. Authorize Drive when prompted.
from google.colab import drive
drive.mount('/content/drive')
bundle_base = '/content/ibpl_mcbyte_urrtcQUzwmo_154_174'
bundle = shutil.make_archive(bundle_base, 'zip', root_dir=RESULTS)
drive_dir = Path('/content/drive/MyDrive/IBPL Video Analysis')
drive_dir.mkdir(parents=True, exist_ok=True)
saved = shutil.copy2(bundle, drive_dir / Path(bundle).name)
print('Saved:', saved)


## What to return
Download or share the generated `ibpl_mcbyte_urrtcQUzwmo_154_174.zip`. It contains the annotated video, MOT tracking records, benchmark report and tracking summary. Keep the full cell error output if installation or tracking fails; it is diagnostic evidence. Do not share the downloaded source-media metadata or temporary signed URLs.
